In [ ]:
import numpy as np 
import random
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

In [ ]:
train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]

In [ ]:
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data


In [ ]:
#define neural network layer
class Layer():
    def __init__(self, n_num, input_dim):
        self.weights = [0]*n_num
        self.weights=self.initialize_parameters(input_dim)
        

    def initialize_parameters(self, input_dim):
        """Initialize parameters with He initialization method"""
        
        parameters = {}
        for l in range(1, len(self.weights)):
            parameters["W"+str(l)] = np.random.randn(self.weights, input_dim) * np.sqrt(2/input_dim)
            parameters["b"+str(l)] = np.zeros((self.weights, 1))

        return parameters


#define fnn    
class FFNN():
    def __init__(self):
        self.layers=0
        #self.weights = []
    
    def add_layer(self, n_num, input_dim=0, activation="relu"):
        if self.layers>0:
            input_dim=len(self.layers[-1])
        new_layer=Layer(n_num, input_dim)
        self.layers.append([new_layer,activation])

    def fit(self, X_train, Y_train, batch_size, epochs=0, learning_rate=0.001):

        self.minibatch_size = batch_size

        for i in range(epochs):
            for batch_idx in range(0, int(X_train.shape[1] / batch_size)):
                # Mini Batch Samples
                if batch_idx == int(X_train.shape[1] / batch_size):
                    X = X_train[:, batch_idx*batch_size:]
                    Y = Y_train[:, batch_idx*batch_size:]
                else:
                    X = X_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]
                    Y = Y_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]

            I = X
            simple_wights=self.feed_forward(I)
            prediction= simple_wights[-1]
            cost=self.cost(prediction, Y)
            gradients =self.backward_prop(simple_wights, Y)
            improved_wights=self.update_parameters(gradients, learning_rate)
        
        self.layers=improved_wights

    
    def feed_forward(self, I):

        forward_vars = {"A0": I}
        for layer in enumerate(self.layers): 
            for w in range(len(layer[0])):
                I = np.dot(I, layer[0]) 
                forward_vars["Z"+str(w)] = np.dot(layer["W"+str(w)], forward_vars["A"+str(w-1)]) + layer["b"+str(w)]
                
                if layer == len(self.layers) - 1: 
                    out_vector = self._activate(I, layer[1]) #output layer 
                else: 
                    I = self._activate(I, layer[1]) #hidden layers 
        #return out_vector

    def backward_prop(self, simple_wights, Y):
        gradients = {}

    def _activate(I, activation):
        if activation=='relu':
            result=np.maximum(0, I)
        elif activation=='softmax':
            T = np.exp(I)
            T_sum = np.sum(T, axis=0)
            result = np.divide(T, T_sum)
        elif activation=='sigmoid':
            result = 1 / (1 + np.exp(-I))
        else:
            print("Error")
        return result
    
    def cost(self, Y_hat, Y):
        """
        Log Loss is applied
        """
        
        m = Y.shape[1]
        if self.binary_classification: 
            cost_value = (1/m) * np.sum(-(Y*np.log(Y_hat) + (1-Y)*np.log(1-Y_hat)))
        else:
            cost_value = (1/m) * np.sum((-1)*Y*np.log(Y_hat))

        return cost_value
    
    def update_parameters(self, parameters, gradients, learning_rate):
        """Update parameters with gradients"""
        
        for l in range(1, len(self.layer_dims)):
            parameters["W"+str(l)] -= learning_rate * gradients["dW"+str(l)]
            parameters["b"+str(l)] -= learning_rate * gradients["db"+str(l)]

        return parameters

In [ ]:
#ceate fnn
fnn_model=FFNN()
fnn_model.add_layer()
fnn_model.fit()


In [ ]:
def create_multiclass_model(input_dim, num_classes):
    model = FFNN()
    model.add_layer(128, input_dim=input_dim, activation='relu') # First hidden layer
    model.add_layer(64, activation='relu')  # Second hidden layer
    model.add_layer(32, activation='relu')  # Third hidden layer
    model.add_layer(num_classes, activation='softmax')  # Output layer for multi-class classification
  
    # Compile the model
    #model.compile(optimizer=Adam(learning_rate=0.001),
    #             loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if using one-hot encoding
    #             metrics=['accuracy'])
    return model

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index], train_data_X[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=train_data_X.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = X_new_reduced[train_index], X_new_reduced[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=X_new_reduced.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy_pca = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy_pca)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy_pca = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")